# StimulusDecode2D

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `decoding_2d`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/StimulusDecode2D.md`


Notebook source link: [StimulusDecode2D.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/StimulusDecode2D.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "StimulusDecode2D"
FAMILY = "decoding_2d"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")


In [ ]:
# 2D Decoding workflow: decode trajectory from place-like tuning fields.
side = 14
grid = np.linspace(0.0, 1.0, side)
gx, gy = np.meshgrid(grid, grid, indexing="xy")
states = np.column_stack([gx.ravel(), gy.ravel()])
n_states = states.shape[0]

n_units = 24
n_time = 280
traj = np.zeros((n_time, 2), dtype=float)
traj[0] = np.array([0.5, 0.5])
vel = np.zeros(2, dtype=float)
for t in range(1, n_time):
    vel = 0.82 * vel + 0.12 * rng.normal(size=2)
    traj[t] = np.clip(traj[t - 1] + vel, 0.0, 1.0)

state_match = np.sum((states[None, :, :] - traj[:, None, :]) ** 2, axis=2)
latent = np.argmin(state_match, axis=1)

centers = rng.uniform(0.0, 1.0, size=(n_units, 2))
sigma = 0.16
dist2 = np.sum((states[None, :, :] - centers[:, None, :]) ** 2, axis=2)
tuning = 0.03 + 0.80 * np.exp(-0.5 * dist2 / (sigma**2))

spike_counts = np.zeros((n_units, n_time), dtype=float)
for t in range(n_time):
    spike_counts[:, t] = rng.poisson(tuning[:, latent[t]])

decoded = DecodingAlgorithms.decode_weighted_center(spike_counts, tuning)
decoded = np.clip(np.rint(decoded), 0, n_states - 1).astype(int)

xy_true = states[latent]
xy_decoded = states[decoded]
rmse = float(np.sqrt(np.mean(np.sum((xy_decoded - xy_true) ** 2, axis=1))))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.5))
axes[0].plot(xy_true[:, 0], xy_true[:, 1], label="true", linewidth=1.2)
axes[0].plot(xy_decoded[:, 0], xy_decoded[:, 1], label="decoded", linewidth=1.0)
axes[0].set_title(f"{TOPIC}: decoded trajectory")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_aspect("equal", adjustable="box")
axes[0].legend(loc="upper right")

field_idx = 6 if TOPIC == "HippocampalPlaceCellExample" else 3
im = axes[1].imshow(
    tuning[field_idx].reshape(side, side),
    origin="lower",
    extent=[0.0, 1.0, 0.0, 1.0],
    cmap="jet",
    aspect="equal",
)
axes[1].set_title("Example receptive field")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
fig.colorbar(im, ax=axes[1], fraction=0.04, pad=0.03)

plt.tight_layout()
plt.show()

print("trajectory rmse", rmse)
assert rmse < 1.25


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
